# 🐍 Python 中的变量：对象、绑定、作用域与复制

> 本笔记用于系统理解 Python 中“变量”的本质，而不仅仅是记忆语法。

## 🎯 学习目标

完成本笔记后，你应该能够解释：

1. Python 中的变量为什么不是“装数据的盒子”；
2. 对象的 `identity`、`type`、`value` 分别是什么；
3. 赋值、重新绑定、修改对象之间的区别；
4. 可变对象与不可变对象的区别；
5. 函数参数为什么有时会影响外部变量，有时不会；
6. 赋值、浅复制、深复制的区别；
7. 为什么 `[[0] * 3] * 3` 会产生三行联动的问题；
8. `global`、`nonlocal` 和 LEGB 作用域规则如何工作。

---

## 📚 权威参考

本笔记主要依据以下资料整理：

- Python 官方文档：Data Model、Execution Model、Simple Statements、Built-in Types
- PEP 8：Style Guide for Python Code
- Luciano Ramalho，《Fluent Python》
- Mark Lutz，《Learning Python》
- Brett Slatkin，《Effective Python》

> 最重要的理解：**Python 变量是名称，名称通过“绑定”引用对象。类型属于对象，不属于变量。**


# 1️⃣ 变量、名称、对象与绑定

在很多语言的入门教学中，变量常被比喻成“盒子”：

```text
变量 a 里面装着数字 10
```

但在 Python 中，更准确的理解是：

```text
名称 a ─────→ 整数对象 10
```

因此：

```python
a = 10
a = "hello world"
```

并不是变量 `a` 的内部从整数变成了字符串，而是：

1. 首先让名称 `a` 绑定到整数对象 `10`；
2. 随后让名称 `a` 改为绑定到字符串对象 `"hello world"`。

```text
第一次赋值：

a ─────→ 10

第二次赋值：

a ─────→ "hello world"
```

这个过程称为 **重新绑定（rebinding）**。


In [ ]:
a = 10
print("第一次赋值：", a, type(a), id(a))

a = "hello world"
print("第二次赋值：", a, type(a), id(a))

# 2️⃣ 对象的三个基本属性

Python 中的数据都以对象形式存在。一个对象最核心的三个属性是：

| 属性 | 英文 | 含义 | 常用查看方式 |
|---|---|---|---|
| 对象身份 | identity | 区分对象“是不是同一个对象” | `id(obj)`、`is` |
| 对象类型 | type | 决定对象支持哪些操作 | `type(obj)`、`isinstance()` |
| 对象的值 | value | 对象表示的数据内容 | 直接查看、`print()`、`==` |

下面逐个理解。


## 2.1 🪪 identity：对象身份

对象身份用于判断两个名称是否引用**同一个对象**。

```python
a = [1, 2]
b = a
```

此时：

```text
a ──┐
    ├────→ [1, 2]
b ──┘
```

`a` 和 `b` 是两个不同的名称，但它们引用的是同一个列表对象。

### 常用工具

- `id(obj)`：返回对象的身份标识；
- `a is b`：判断 `a` 和 `b` 是否引用同一个对象；
- `a is not b`：判断是否不是同一个对象。

> 在 CPython 中，`id()` 通常与对象的内存地址有关，但 Python 语言层面只保证：对象存活期间，其 `id` 唯一且不变。不要依赖具体内存地址实现程序逻辑。


In [ ]:
a = [1, 2]
b = a
c = [1, 2]

print("id(a):", id(a))
print("id(b):", id(b))
print("id(c):", id(c))

print("a is b:", a is b)   # 同一个对象
print("a is c:", a is c)   # 不同对象
print("a == c:", a == c)   # 值相等

### `is` 与 `==` 的区别

| 运算符 | 判断内容 | 示例 |
|---|---|---|
| `is` | 是否为同一个对象 | `a is b` |
| `==` | 对象的值是否相等 | `a == b` |

```python
a = [1, 2]
b = [1, 2]
```

虽然两个列表内容一样，但它们是分别创建的两个对象：

```text
a ─────→ [1, 2]

b ─────→ [1, 2]
```

因此：

```python
a == b   # True
a is b   # False
```

通常只有判断 `None` 等单例对象时使用 `is`：

```python
if result is None:
    ...
```


## 2.2 🧩 type：对象类型

对象类型决定：

- 对象支持哪些操作；
- 对象内部如何表示数据；
- 对象是可变还是不可变；
- 运算符对它产生什么效果。

例如：

```python
10 + 20
```

整数支持数值加法。

```python
"hello" + " world"
```

字符串的 `+` 表示拼接。

```python
[1, 2] + [3, 4]
```

列表的 `+` 表示创建一个新的拼接列表。

### 常用工具

```python
type(obj)
isinstance(obj, SomeType)
```

通常业务代码中更推荐使用 `isinstance()`，因为它支持继承关系。


In [ ]:
values = [
    10,
    3.14,
    "Python",
    [1, 2],
    (1, 2),
    {"name": "Tom"},
    {1, 2, 3},
    None,
]

for value in values:
    print(f"{value!r:20} -> {type(value).__name__}")

## 2.3 💎 value：对象的值

对象的值是对象所表示的数据内容。

```python
a = [1, 2]
b = [1, 2]
```

虽然 `a` 和 `b` 的身份不同，但值相等：

```python
a is b   # False
a == b   # True
```

对于可变对象，值可以在对象身份不变的情况下发生变化：

```python
numbers = [1, 2]
before = id(numbers)

numbers.append(3)

after = id(numbers)
```

列表内容从 `[1, 2]` 变为 `[1, 2, 3]`，但仍然是原来的列表对象。


In [ ]:
numbers = [1, 2]
before_id = id(numbers)

numbers.append(3)
after_id = id(numbers)

print("修改后的值：", numbers)
print("修改前 id：", before_id)
print("修改后 id：", after_id)
print("是否仍为同一对象：", before_id == after_id)

## 2.4 三个属性怎样一起使用？

在调试代码时，可以连续观察对象的三个属性：

```python
print("身份：", id(obj))
print("类型：", type(obj))
print("值：", obj)
```

例如，对比“修改对象”和“重新绑定”：

| 操作 | identity | type | value |
|---|---|---|---|
| `numbers.append(3)` | 通常不变 | 不变 | 改变 |
| `numbers = [100]` | 改变 | 可能相同 | 改变 |
| `x = 10; x += 1` | 改变 | 不变 | 改变 |

这张表是理解 Python 变量的关键。


In [ ]:
# 修改可变对象：identity 不变，value 改变
numbers = [1, 2]
print("修改前：", id(numbers), type(numbers), numbers)

numbers.append(3)
print("修改后：", id(numbers), type(numbers), numbers)

print("-" * 50)

# 重新绑定：名称改为指向另一个对象
numbers = [100]
print("重新绑定后：", id(numbers), type(numbers), numbers)

# 3️⃣ 可变对象与不可变对象

## 常见不可变对象

- `int`
- `float`
- `bool`
- `complex`
- `str`
- `tuple`
- `bytes`
- `frozenset`
- `range`
- `None`

不可变对象创建后，不能原地改变其值。

```python
x = 10
x += 1
```

这里不是把整数对象 `10` 修改成 `11`，而是：

1. 计算出一个新的整数对象 `11`；
2. 让名称 `x` 改为绑定到 `11`。

## 常见可变对象

- `list`
- `dict`
- `set`
- `bytearray`
- 大多数自定义类实例

它们通常可以在身份不变的情况下修改值。


In [ ]:
# 不可变对象：重新绑定
x = 10
y = x

print("修改前：")
print("x =", x, "id =", id(x))
print("y =", y, "id =", id(y))

x += 1

print("\n修改后：")
print("x =", x, "id =", id(x))
print("y =", y, "id =", id(y))

# 4️⃣ Python 的赋值是名称绑定

```python
a = [1, 2]
b = a
```

赋值 `b = a` 不会复制列表，只会让名称 `b` 绑定到 `a` 当前引用的对象。

```text
a ──┐
    ├────→ [1, 2]
b ──┘
```

因此通过任意一个名称修改该对象，另一个名称都能看到变化。


In [ ]:
a = [1, 2]
b = a

b.append(3)

print(a)  # [1, 2, 3]
print(b)  # [1, 2, 3]
print("a is b:", a is b)

# 5️⃣ Python 变量的命名规则

名称可以包含：

- 字母；
- 数字；
- 下划线 `_`；
- 符合 Python 标识符规则的 Unicode 字符。

但需要满足：

1. 不能以数字开头；
2. 不能使用 Python 关键字；
3. 区分大小写。

```python
student_name = "Tom"   # ✅
score2 = 95            # ✅
_private = 10          # ✅
姓名 = "张三"           # ✅

# 2score = 95          # ❌
# class = "A"          # ❌
```

## PEP 8 常见命名约定

| 对象 | 推荐形式 | 示例 |
|---|---|---|
| 普通变量 | 小写下划线 | `student_name` |
| 函数 | 小写下划线 | `calculate_score()` |
| 类 | 大驼峰 | `StudentRecord` |
| 常量 | 全大写下划线 | `MAX_RETRY_COUNT` |
| 内部使用名称 | 单前导下划线 | `_cache` |

避免覆盖内置名称：

```python
list = [1, 2]   # 不推荐
str = "hello"   # 不推荐
sum = 100       # 不推荐
```


# 6️⃣ 常见的名称绑定方式

## 6.1 普通赋值

```python
age = 20
name = "Tom"
```

## 6.2 链式赋值

```python
a = b = c = 0
```

三个名称绑定到同一个整数对象。

对于不可变对象通常没有明显问题：

```python
a = b = 0
a += 1
```

`a` 会重新绑定到 `1`，`b` 仍绑定到 `0`。

但可变对象要特别小心：

```python
a = b = []
a.append(1)
```

因为 `a` 和 `b` 引用同一个列表。

需要两个独立列表时，应分别创建：

```python
a = []
b = []
```


In [ ]:
a = b = []

a.append(1)

print("a:", a)
print("b:", b)
print("a is b:", a is b)

## 6.3 多变量赋值与解包

```python
x, y = 10, 20
```

右侧先构造或计算出一组值，再分别绑定到左侧名称。

交换变量：

```python
x, y = y, x
```

右侧会先求值，因此不需要临时变量。

## 6.4 嵌套解包

```python
name, (math_score, english_score) = "Tom", (95, 88)
```

结构必须匹配。

## 6.5 星号解包

```python
first, *middle, last = [1, 2, 3, 4, 5]
```

结果：

```python
first == 1
middle == [2, 3, 4]
last == 5
```

带星号的名称会接收剩余元素，并得到一个列表。


In [ ]:
x, y = 10, 20
print("交换前：", x, y)

x, y = y, x
print("交换后：", x, y)

name, (math_score, english_score) = "Tom", (95, 88)
print(name, math_score, english_score)

first, *middle, last = [1, 2, 3, 4, 5]
print(first)
print(middle)
print(last)

# 7️⃣ 增强赋值

常见增强赋值运算符：

```text
+=  -=  *=  /=  //=  %=  **=
&=  |=  ^=  <<=  >>=
```

增强赋值需要结合对象类型理解。

## 不可变对象

```python
x = 10
x += 1
```

整数无法原地修改，因此会创建新整数并重新绑定 `x`。

## 可变对象

```python
numbers = [1, 2]
numbers += [3]
```

列表的 `+=` 通常会原地修改原列表，效果接近：

```python
numbers.extend([3])
```

因此：

```python
numbers += [3]
```

与：

```python
numbers = numbers + [3]
```

可能产生不同的引用效果。


In [ ]:
# += 通常原地修改列表
a = [1, 2]
b = a
a += [3]

print("使用 += 后：")
print("a:", a)
print("b:", b)
print("a is b:", a is b)

print("-" * 50)

# + 创建新列表，然后重新绑定
a = [1, 2]
b = a
a = a + [3]

print("使用 a = a + [3] 后：")
print("a:", a)
print("b:", b)
print("a is b:", a is b)

# 8️⃣ 海象运算符 `:=`

赋值表达式 `:=` 可以在表达式内部绑定名称，同时返回该值。

```python
if length := len("Python"):
    print(length)
```

相当于：

```python
length = len("Python")
if length:
    print(length)
```

典型使用场景：

```python
while chunk := file.read(1024):
    process(chunk)
```

⚠️ 不要为了少写一行代码而滥用，否则可能降低可读性。


In [ ]:
if length := len("Python"):
    print("字符串长度：", length)

# 9️⃣ 属性赋值与下标赋值

## 属性赋值

```python
student.name = "Tom"
```

这里不是给当前作用域创建普通变量 `name`，而是在 `student` 对象上设置或修改属性。

## 下标赋值

```python
numbers[0] = 100
student["name"] = "Tom"
```

这里修改的是容器中的元素。

| 形式 | 修改目标 |
|---|---|
| `x = value` | 名称 `x` 的绑定 |
| `obj.attr = value` | 对象的属性 |
| `container[index] = value` | 容器中的元素 |


# 🔟 作用域与 LEGB

Python 查找普通名称时，可以用 LEGB 模型理解：

| 缩写 | 名称 | 含义 |
|---|---|---|
| L | Local | 当前函数局部作用域 |
| E | Enclosing | 外层函数作用域 |
| G | Global | 当前模块全局作用域 |
| B | Built-in | Python 内置作用域 |

查找顺序通常是：

```text
Local → Enclosing → Global → Built-in
```


In [ ]:
message = "global"

def outer():
    message = "enclosing"

    def inner():
        message = "local"
        print(message)

    inner()

outer()

## 10.1 `global`

模块顶层定义的名称属于全局作用域：

```python
count = 0
```

函数可以直接读取全局变量：

```python
def show():
    print(count)
```

但要在函数内部重新绑定全局名称，需要使用 `global`：

```python
def increase():
    global count
    count += 1
```

## 10.2 `nonlocal`

嵌套函数需要重新绑定外层函数中的变量时，使用 `nonlocal`：

```python
def outer():
    count = 0

    def inner():
        nonlocal count
        count += 1
```

区别：

| 关键字 | 指向的作用域 |
|---|---|
| `global` | 当前模块全局作用域 |
| `nonlocal` | 最近的外层函数作用域 |


In [ ]:
count = 0

def increase_global():
    global count
    count += 1

increase_global()
print("全局 count：", count)


def make_counter():
    count = 0

    def increase():
        nonlocal count
        count += 1
        return count

    return increase

counter = make_counter()
print(counter())
print(counter())
print(counter())

# 1️⃣1️⃣ 函数参数也是局部变量

考虑：

```python
def greet(name):
    print(name)
```

参数 `name` 是函数的局部名称。

调用：

```python
greet("Tom")
```

可以理解为：进入函数时，让局部名称 `name` 绑定到传入的字符串对象。

Python 的参数传递常被描述为：

- 按赋值传递；
- 对象共享传递；
- call by sharing。

核心规则是：

> 调用函数时，形参名称会绑定到实参当前引用的对象。

函数内部随后既可以：

1. 重新绑定形参；
2. 修改形参引用的可变对象。

这两种操作的结果不同。


## 11.1 重新绑定参数：外部变量不受影响

```python
def rebind(value):
    value = [100]

numbers = [1, 2]
rebind(numbers)
```

进入函数时：

```text
numbers ──┐
          ├────→ [1, 2]
value ────┘
```

执行：

```python
value = [100]
```

之后：

```text
numbers ─────→ [1, 2]

value ───────→ [100]
```

只改变了函数内部局部名称 `value` 的绑定，没有修改原列表。


In [ ]:
def rebind(value):
    print("函数内，重新绑定前：", id(value), value)
    value = [100]
    print("函数内，重新绑定后：", id(value), value)

numbers = [1, 2]
print("函数外，调用前：", id(numbers), numbers)

rebind(numbers)

print("函数外，调用后：", id(numbers), numbers)

## 11.2 修改共享对象：外部变量会看到变化

```python
def mutate(value):
    value.append(100)

numbers = [1, 2]
mutate(numbers)
```

进入函数时：

```text
numbers ──┐
          ├────→ [1, 2]
value ────┘
```

执行：

```python
value.append(100)
```

不是让 `value` 指向新对象，而是修改双方共同引用的列表：

```text
numbers ──┐
          ├────→ [1, 2, 100]
value ────┘
```

因此函数外的 `numbers` 也能看到变化。


In [ ]:
def mutate(value):
    print("函数内，修改前：", id(value), value)
    value.append(100)
    print("函数内，修改后：", id(value), value)

numbers = [1, 2]
print("函数外，调用前：", id(numbers), numbers)

mutate(numbers)

print("函数外，调用后：", id(numbers), numbers)

## 11.3 重新绑定与修改对象对照

| 语句 | 本质 | 是否可能影响外部列表 |
|---|---|---|
| `value = [100]` | 重新绑定局部名称 | 否 |
| `value = value + [100]` | 创建新列表并重新绑定 | 否 |
| `value.append(100)` | 修改原列表 | 是 |
| `value[0] = 100` | 修改原列表中的元素 | 是 |
| `value.clear()` | 清空原列表 | 是 |
| `value += [100]` | 列表通常原地扩展 | 是 |

判断方法：

> 看代码是在“改变箭头”，还是在“修改箭头指向的对象”。


In [ ]:
def use_plus(value):
    value = value + [100]

def use_plus_equal(value):
    value += [100]

a = [1, 2]
use_plus(a)
print("使用 value = value + [100]：", a)

b = [1, 2]
use_plus_equal(b)
print("使用 value += [100]：", b)

# 1️⃣2️⃣ 赋值、浅复制与深复制

先构造一个嵌套列表：

```python
a = [[1, 2], [3, 4]]
```

这里实际有三个列表对象：

```text
a
│
▼
外层列表
[ 引用A, 引用B ]
    │       │
    ▼       ▼
 [1, 2]   [3, 4]
```

## 什么是“最外层容器”？

在：

```python
[[1, 2], [3, 4]]
```

中：

- 整个 `[[1, 2], [3, 4]]` 是最外层列表；
- `[1, 2]` 和 `[3, 4]` 是内层列表；
- 整数 `1、2、3、4` 是内层列表中的元素。

可以把它想成：

```text
外层列表 = 一个书架
内层列表 = 书架上的抽屉
整数对象 = 抽屉里的物品
```


## 12.1 赋值：完全不复制对象

```python
b = a
```

结构：

```text
a ──┐
    ├────→ 同一个外层列表
b ──┘
```

外层列表和内部列表全部共享。

```python
a is b       # True
a[0] is b[0] # True
```


In [ ]:
a = [[1, 2], [3, 4]]
b = a

print("a is b:", a is b)
print("a[0] is b[0]:", a[0] is b[0])

b.append([5, 6])
b[0].append(100)

print("a:", a)
print("b:", b)

## 12.2 浅复制：只复制最外层容器

```python
b = a.copy()
```

也可以写：

```python
b = list(a)
b = a[:]
```

结构：

```text
a ─→ 外层列表 A ─→ 内层列表 1
                └→ 内层列表 2

b ─→ 外层列表 B ─→ 内层列表 1
                └→ 内层列表 2
```

因此：

```python
a is b        # False：外层不同
a[0] is b[0] # True：内层仍共享
```

### 修改外层结构

```python
b.append([5, 6])
```

只影响 `b`，因为 `a` 和 `b` 的外层列表不同。

### 修改内层对象

```python
b[0].append(100)
```

会同时出现在 `a` 和 `b` 中，因为二者共享第一个内层列表。


In [ ]:
a = [[1, 2], [3, 4]]
b = a.copy()

print("a is b:", a is b)
print("a[0] is b[0]:", a[0] is b[0])

b.append([5, 6])
print("\n修改 b 的外层结构后：")
print("a:", a)
print("b:", b)

b[0].append(100)
print("\n修改共享的内层列表后：")
print("a:", a)
print("b:", b)

## 12.3 深复制：递归复制内部对象

```python
import copy
b = copy.deepcopy(a)
```

结构：

```text
a ─→ 外层列表 A ─→ 内层列表 A1
                └→ 内层列表 A2

b ─→ 外层列表 B ─→ 内层列表 B1
                └→ 内层列表 B2
```

通常：

```python
a is b         # False
a[0] is b[0]  # False
```

修改 `b` 的内层列表，不会影响 `a`。


In [ ]:
import copy

a = [[1, 2], [3, 4]]
b = copy.deepcopy(a)

print("a is b:", a is b)
print("a[0] is b[0]:", a[0] is b[0])

b[0].append(100)

print("a:", a)
print("b:", b)

## 12.4 三种方式总表

| 操作 | 是否创建新外层容器 | 内层对象是否共享 | 常见用途 |
|---|---:|---:|---|
| `b = a` | ❌ | ✅ 全部共享 | 需要两个名称引用同一对象 |
| `b = a.copy()` | ✅ | ✅ 仍共享 | 只需独立修改外层结构 |
| `copy.deepcopy(a)` | ✅ | ❌ 通常递归复制 | 需要完全独立的嵌套结构 |

⚠️ 深复制并不是越多越好：

- 可能消耗更多内存和时间；
- 某些资源对象不能合理复制；
- 自定义类可以控制复制行为；
- 如果对象中存在共享关系，`deepcopy` 会尽量保留这种结构，而不是无限重复复制。


In [ ]:
import copy

a = [[1, 2], [3, 4]]
assignment = a
shallow = a.copy()
deep = copy.deepcopy(a)

print("外层是否相同：")
print("a is assignment:", a is assignment)
print("a is shallow:", a is shallow)
print("a is deep:", a is deep)

print("\n第一层内部对象是否相同：")
print("a[0] is assignment[0]:", a[0] is assignment[0])
print("a[0] is shallow[0]:", a[0] is shallow[0])
print("a[0] is deep[0]:", a[0] is deep[0])

# 1️⃣3️⃣ 二维列表陷阱：`[[0] * 3] * 3`

代码：

```python
matrix = [[0] * 3] * 3
```

先拆开：

```python
row = [0] * 3
matrix = [row] * 3
```

第一步创建一个列表：

```text
row ─────→ [0, 0, 0]
```

第二步并没有创建三个独立行，而是把同一个 `row` 的引用重复三次：

```text
matrix
  │
  ▼
[ 引用1, 引用2, 引用3 ]
    │      │      │
    └──────┼──────┘
           ▼
       [0, 0, 0]
```

因此：

```python
matrix[0] is matrix[1] is matrix[2]
```

结果为 `True`。

修改：

```python
matrix[0][0] = 1
```

实际修改的是三行共同引用的那个列表，所以打印时三行都会显示变化。


In [ ]:
matrix = [[0] * 3] * 3

print("三行是否是同一个对象：")
print(matrix[0] is matrix[1])
print(matrix[1] is matrix[2])

matrix[0][0] = 1
print("修改后的矩阵：", matrix)

## 正确创建三行独立列表

```python
matrix = [[0] * 3 for _ in range(3)]
```

列表推导式每循环一次，都会执行一次：

```python
[0] * 3
```

因此会创建三个独立列表：

```text
matrix
  │
  ▼
[ 引用1, 引用2, 引用3 ]
    │      │      │
    ▼      ▼      ▼
[0,0,0] [0,0,0] [0,0,0]
```


In [ ]:
matrix = [[0] * 3 for _ in range(3)]

print("三行是否是同一个对象：")
print(matrix[0] is matrix[1])
print(matrix[1] is matrix[2])

matrix[0][0] = 1
print("修改后的矩阵：", matrix)

# 1️⃣4️⃣ 常见变量陷阱

## 14.1 可变默认参数

错误写法：

```python
def add_item(item, items=[]):
    items.append(item)
    return items
```

默认参数对象通常只在函数定义时创建一次，因此多次调用会共享同一个列表。

正确写法：

```python
def add_item(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items
```

## 14.2 用 `is` 比较普通数值或字符串

错误思路：

```python
a is b
```

判断值是否相等应使用：

```python
a == b
```

`is` 用于判断对象身份。

## 14.3 覆盖内置名称

```python
list = [1, 2, 3]
sum = 100
```

这会使同一作用域中无法正常使用原内置对象。

## 14.4 误以为赋值会复制对象

```python
backup = original
```

这不是备份，只是创建另一个引用。


In [ ]:
def bad_add_item(item, items=[]):
    items.append(item)
    return items

print(bad_add_item("A"))
print(bad_add_item("B"))
print(bad_add_item("C"))

# 1️⃣5️⃣ 一套判断代码影响范围的方法

看到变量相关代码时，可以依次问：

### 第一步：名称指向什么对象？

```python
type(obj)
id(obj)
```

### 第二步：这是重新绑定，还是修改对象？

重新绑定：

```python
x = new_object
```

修改对象：

```python
x.append(...)
x[key] = ...
x.attr = ...
```

### 第三步：对象是可变还是不可变？

- 不可变对象不能原地改变；
- 可变对象可能被多个名称共享。

### 第四步：是否存在嵌套容器？

```python
[[1, 2], [3, 4]]
```

要分别检查：

```python
a is b
a[0] is b[0]
```

### 第五步：是在函数内部吗？

查看形参是在：

- 重新绑定；
- 还是修改传入的共享对象。


# 1️⃣6️⃣ 综合示例

下面的代码同时展示：

- 名称绑定；
- 对象身份；
- 重新绑定；
- 修改对象；
- 浅复制；
- 深复制。


In [ ]:
import copy

original = [[1, 2], [3, 4]]

alias = original
shallow = original.copy()
deep = copy.deepcopy(original)

print("初始状态：")
print("original:", original)
print("alias   :", alias)
print("shallow :", shallow)
print("deep    :", deep)

print("\n修改 original[0].append(100)")
original[0].append(100)

print("original:", original)
print("alias   :", alias)
print("shallow :", shallow)
print("deep    :", deep)

print("\n给 original 重新绑定新列表")
original = [["new"]]

print("original:", original)
print("alias   :", alias)
print("shallow :", shallow)
print("deep    :", deep)

# 1️⃣7️⃣ 自测题 🧠

请先自己判断，再运行代码。

## 题目 1

```python
a = [1, 2]
b = a
b = [100]

print(a)
```

## 题目 2

```python
a = [[1], [2]]
b = a.copy()
b[0].append(100)

print(a)
print(b)
```

## 题目 3

```python
def change(data):
    data[0] = 100
    data = [200]

numbers = [1, 2]
change(numbers)

print(numbers)
```

## 题目 4

```python
matrix = [[0] * 2] * 2
matrix[1].append(9)

print(matrix)
```


# ✅ 自测题答案

## 题目 1

```python
[1, 2]
```

`b = [100]` 只是让 `b` 重新绑定到新列表，不修改 `a` 指向的列表。

## 题目 2

```python
[[1, 100], [2]]
[[1, 100], [2]]
```

浅复制只复制外层列表，`a[0]` 与 `b[0]` 仍是同一个内层列表。

## 题目 3

```python
[100, 2]
```

`data[0] = 100` 修改共享列表，因此影响 `numbers`；随后 `data = [200]` 只是重新绑定局部名称。

## 题目 4

```python
[[0, 0, 9], [0, 0, 9]]
```

两行是同一个列表对象。


# 1️⃣8️⃣ 最终知识框架

```text
名称（变量）
    │
    │ 绑定
    ▼
对象
├── identity：是不是同一个对象
├── type：对象支持什么操作
├── value：对象表示什么数据
└── mutability：对象是否可以原地修改
```

进一步理解：

```text
赋值
├── 普通赋值：建立绑定
├── 重新绑定：名称改为指向新对象
├── 修改对象：对象值改变，身份可能不变
├── 浅复制：复制最外层容器
└── 深复制：递归复制嵌套对象
```

函数参数：

```text
调用函数
    ↓
形参绑定到实参引用的对象
    ├── 重新绑定形参：通常不影响调用者
    └── 修改共享可变对象：调用者可以看到变化
```

## 🌟 一句话总结

> Python 变量不是装数据的盒子，而是指向对象的名称。理解“名称绑定、对象共享、重新绑定、原地修改”，就理解了 Python 变量的大部分行为。


# 🔗 参考资料

## Python 官方文档

- Data Model  
  https://docs.python.org/3/reference/datamodel.html
- Execution Model  
  https://docs.python.org/3/reference/executionmodel.html
- Simple Statements  
  https://docs.python.org/3/reference/simple_stmts.html
- Built-in Types  
  https://docs.python.org/3/library/stdtypes.html
- `copy` — Shallow and deep copy operations  
  https://docs.python.org/3/library/copy.html
- PEP 8 — Style Guide for Python Code  
  https://peps.python.org/pep-0008/

## 教材

- Luciano Ramalho, *Fluent Python*
- Mark Lutz, *Learning Python*
- Brett Slatkin, *Effective Python*
